# Retrieval for RAG with LangChain

One policy handbook (`heliodesk-policies.md`), one question, and each retrieval idea added step by step.

1. Embeddings
2. Chunking
3. The vector database
4. Naive RAG
5. Reranking
6. BM25 (keyword search)
7. Hybrid search
8. Query rewriting (HyDE)

Same models as the course, through **OpenRouter** (Gemini embeddings, Cohere reranker, Gemini chat model) and **ChromaDB**. This version uses **LangChain**, so most blocks are one ready-made class:

| Job | LangChain class |
|---|---|
| Embeddings | `OpenAIEmbeddings` |
| Chat model | `ChatOpenAI` |
| Split the handbook | `RecursiveCharacterTextSplitter` |
| Vector database | `Chroma` |
| Keyword search | `BM25Retriever` |
| Hybrid search | `EnsembleRetriever` |
| Query rewriting | `HypotheticalDocumentEmbedder` |

In [ ]:
%pip install -q langchain-classic langchain-openai langchain-chroma langchain-community langchain-text-splitters chromadb rank_bm25 httpx python-dotenv pandas numpy

# 1. Embeddings

An embedding turns text into a list of numbers. Texts with a similar **meaning** get similar numbers.

We get embeddings from OpenRouter. The key is read from a `.env` file next to the notebook:

```
OPENROUTER_API_KEY=sk-or-...
```

In Colab you can add it as a secret named `OPENROUTER_API_KEY` instead.

In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    from google.colab import userdata      # Colab Secrets
    API_KEY = userdata.get("OPENROUTER_API_KEY")

BASE_URL = "https://openrouter.ai/api/v1"

embeddings = OpenAIEmbeddings(
    model="google/gemini-embedding-001",
    base_url=BASE_URL,
    api_key=API_KEY,
    check_embedding_ctx_length=False,     # send plain text, not OpenAI token ids
)

vector = embeddings.embed_query("Never paste the download link into the ticket thread.")
print(len(vector), "numbers")
print(vector[:8])

3072 numbers
[-0.01890663616359234, 0.015278679318726063, -0.028450857847929, -0.06917754560709, 0.02044777199625969, 0.011158215813338757, 0.014934467151761055, -0.017426230013370514]


Cosine similarity compares two vectors: close to 1 means similar meaning.

Sentences 0 and 1 say the same thing in different words, and so do 2 and 3. Sentence 4 is unrelated.

In [2]:
sentences = [
    "Files with personal data must not be copied into ticket comments.",
    "Never paste the download link into the ticket thread.",
    "Annual-plan customers may request a refund review within 14 days of renewal.",
    "Can I get my money back after the yearly renewal?",
    "Password resets are routed to the identity-provider administrator.",
    "Bachata is danced from side to side",
    "Salsa is danced on 1 and 5"
]

vectors = np.array(embeddings.embed_documents(sentences))
vectors

array([[ 0.02044289,  0.04233876, -0.00035352, ..., -0.01975122,
        -0.0194135 , -0.0108998 ],
       [-0.01890664,  0.01527868, -0.02845086, ..., -0.01568687,
        -0.0113511 , -0.0044524 ],
       [-0.02299296,  0.01193523, -0.02078801, ..., -0.02296024,
        -0.02189561, -0.00980838],
       ...,
       [ 0.01279441,  0.02985772, -0.00652729, ..., -0.04246346,
         0.01092184, -0.01842536],
       [ 0.02724222,  0.01869259, -0.01102935, ...,  0.00826869,
        -0.0139943 ,  0.01101111],
       [-0.00388301,  0.02800911, -0.01442297, ..., -0.00335437,
        -0.00806724, -0.00129481]], shape=(7, 3072))

In [3]:

vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)   # length 1, so a dot product is the cosine
similarity = vectors @ vectors.T

pd.DataFrame(similarity.round(2), index=range(7), columns=range(7))

,0,1,2,3,4,5,6
0,1.00,0.76,0.54,0.51,0.56,0.47,0.45
1,0.76,1.00,0.57,0.56,0.57,0.46,0.46
2,0.54,0.57,1.00,0.79,0.55,0.43,0.45
3,0.51,0.56,0.79,1.00,0.51,0.46,0.47
4,0.56,0.57,0.55,0.51,1.00,0.44,0.45
5,0.47,0.46,0.43,0.46,0.44,1.00,0.69
6,0.45,0.46,0.45,0.47,0.45,0.69,1.00


# 2. Chunking

We split the handbook into pieces. Each piece will become one vector.

- **chunk size**: how long each piece is
- **overlap**: how much neighbouring pieces can share, so a sentence cut at the border still appears whole somewhere

The splitter tries the biggest natural boundary first (paragraph, then line, then space), so overlap only appears when it has to cut inside a paragraph.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

#POLICY_URL = "https://raw.githubusercontent.com/richhiey/ai-app-dev_Mod-A/main/data/heliodesk-policies.md"

path = Path("heliodesk-policies.md")
text = path.read_text(encoding="utf-8") if path.exists() else urlopen(POLICY_URL).read().decode("utf-8")
print(len(text), "characters")

14400 characters


In [6]:
text

"# HelioDesk support policy handbook\n\nThis handbook is the source document for HelioDesk support retrieval practice. It collects operating policies that support specialists must follow when answering customer questions about exports, account access, refunds, invoices, Enterprise commitments, data sharing, trials, security, and internal support notes. The document is intentionally written as normal policy text, not as pre-made retrieval chunks. The notebook loads this document, splits it with the shared `chunk_text(...)` helper, and indexes the generated chunks in ChromaDB.\n\nSupport specialists should answer from the policy evidence that is actually retrieved. If the retrieved context does not contain enough evidence for a decision, the specialist should say what is known, explain what is missing, and route the case to the correct internal owner. Support should not invent exceptions, promise timelines that are not written in policy, or cite internal-only controls in customer-facing 

In [7]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=850, chunk_overlap=120)

# LangChain works with Document objects: the text plus metadata (here, the chunk number)
chunks = [
    Document(page_content=piece, metadata={"chunk": i})
    for i, piece in enumerate(splitter.split_text(text))
]

print(len(chunks), "chunks")
print()
print(chunks[3].page_content)

22 chunks

Before starting an export, support must confirm the workspace slug, requester email, and business reason. If the request comes from a contractor, auditor, reseller, or former employee, support must verify that the current account owner has authorized the request. The internal note should record who requested the export, which workspace was selected, and which verification step was completed.

Only workspace administrators and billing managers can download completed export files. Standard support agents may create the export request for an eligible account, but they must ask an administrator or billing manager to download and share the file. If the customer asks for external delivery, support should use the secure customer portal rather than sending the archive as an email attachment.


In [ ]:
chunks

# 3. The vector database

A vector database stores each chunk's **text** and **vector**, and finds the nearest vectors to a question.

We give Chroma the embedding model, so it creates the vectors itself, both when we store the chunks and when we search.

In [8]:
import chromadb
from langchain_chroma import Chroma

chroma_client = chromadb.EphemeralClient()

vectorstore = Chroma(
    client=chroma_client,
    collection_name="heliodesk",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)

vectorstore.add_documents(chunks, ids=[str(c.metadata["chunk"]) for c in chunks])

print(len(vectorstore.get()["ids"]), "chunks stored")

22 chunks stored


# 4. Naive RAG

**question → embed → nearest chunks → paste them into the prompt → the LLM answers**

Our question. A good answer needs:
- only workspace administrators and billing managers can download the export
- support must verify written authorization from the account owner
- delivery goes through the secure customer portal
- the link must not be pasted into ticket comments

In [9]:
FOCUS_QUESTION = (
    "A customer's outside reviewer is doing a quarterly controls check and asks the support agent "
    "for the workspace records package. The agent wants to paste a download link into the ticket "
    "thread so the reviewer can grab the file. What should support verify before anything is "
    "shared, and what is the approved delivery path?"
)

CANDIDATE_K = 10        # how many chunks the first search returns
FINAL_CONTEXT_K = 3     # how many chunks go to the LLM

# a retriever turns a question into a list of the nearest chunks
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": CANDIDATE_K})


def show(docs):
    for rank, doc in enumerate(docs, 1):
        print(f"{rank:>2}. chunk {doc.metadata['chunk']:>2} | {doc.page_content[:90]!r}")


candidates = vector_retriever.invoke(FOCUS_QUESTION)
show(candidates)

 1. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 2. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'
 3. chunk  2 | '## Customer data exports\n\nSupport agents can request a customer data export from the works'
 4. chunk 15 | 'Legal requests, subpoenas, and law-enforcement requests must move to Legal Operations. Sup'
 5. chunk  4 | 'Completed customer data export files remain available for 30 days after creation. After th'
 6. chunk  1 | 'Support specialists should answer from the policy evidence that is actually retrieved. If '
 7. chunk 17 | 'If a support specialist accidentally exposes customer data in a ticket, macro, screenshot,'
 8. chunk 21 | 'Deletion requests that mention legal dispute, security incident, chargeback, or audit inve'
 9. chunk 16 | '## Security incident handling\n\nIf a customer reports suspected account compromise, unusual'
10. chunk 18 | 'When a

In [12]:
vector_retriever.invoke("the account will be closed")

[Document(id='21', metadata={'chunk': 21}, page_content='Deletion requests that mention legal dispute, security incident, chargeback, or audit investigation require escalation before the account is closed. Support should not complete a deletion request while a legal hold or active security investigation is attached to the workspace. If the policy evidence is unclear, the safer response is to route the case rather than promise deletion.'),
 Document(id='20', metadata={'chunk': 20}, page_content='Plan downgrade requests take effect at the next renewal unless billing operations approves an exception. Support should explain feature loss, export recommendations, and account-owner approval requirements before routing a downgrade request. If the workspace has Enterprise commitments, the success contact should be notified before downgrade guidance is sent.\n\n## Deletion, retention, and account closure\n\nAccount closure requests must be confirmed by the account owner. Support should tell the 

Only the top 3 go into the prompt.

In [14]:
naive_context = candidates[:FINAL_CONTEXT_K]
show(naive_context)

 1. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 2. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'
 3. chunk  2 | '## Customer data exports\n\nSupport agents can request a customer data export from the works'


In [43]:
naive_context

Now the LLM: a prompt, the model, and an output parser, chained with `|`.

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="google/gemini-2.5-flash-lite", base_url=BASE_URL, api_key=API_KEY, temperature=0)

answer_chain = (
    ChatPromptTemplate.from_template(
        "Answer the question using only the policy context."
        "If the context is not enough, say what is missing.\n\n"
        "Context:\n{context}\n\nQuestion: {question}"
    )
    | llm
    | StrOutputParser()
)


def answer(question, docs):
    context = "\n\n".join(f"[chunk {d.metadata['chunk']}] {d.page_content}" for d in docs)
    return answer_chain.invoke({"context": context, "question": question})


print(answer(FOCUS_QUESTION, naive_context))

Before anything is shared, support must verify written authorization from the account owner. This authorization note should name the auditor (in this case, the outside reviewer), the requested date range, the business reason for the request, and the person who approved access.

The approved delivery path is through the secure customer portal. Files containing personal data must not be shared through public links or copied into ticket comments. Support should use the portal link, expiry controls, and audit note required by the data-sharing workflow.


# 5. Reranking

The first search is fast but rough. A **reranker** reads the question together with each candidate and scores how well that chunk answers it.

**search 10 candidates → rerank them → keep the best 3**

LangChain's Cohere reranker class is built for Cohere's own API and key. Ours goes through OpenRouter, so this one step is a plain HTTP call.

In [16]:
import httpx


def rerank(query, docs, top_n=FINAL_CONTEXT_K):
    response = httpx.post(
        f"{BASE_URL}/rerank",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "model": "cohere/rerank-v3.5",
            "query": query,
            "documents": [d.page_content for d in docs],
            "top_n": top_n,
        },
        timeout=60,
    )
    response.raise_for_status()
    return [docs[item["index"]] for item in response.json()["results"]]


reranked = rerank(FOCUS_QUESTION, candidates)

print("BEFORE (top 3)")
show(naive_context)
print("\nAFTER reranking (top 3)")
show(reranked)

BEFORE (top 3)
 1. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 2. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'
 3. chunk  2 | '## Customer data exports\n\nSupport agents can request a customer data export from the works'

AFTER reranking (top 3)
 1. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'
 2. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 3. chunk 15 | 'Legal requests, subpoenas, and law-enforcement requests must move to Legal Operations. Sup'


In [17]:
print(answer(FOCUS_QUESTION, reranked))

Before anything is shared, support must verify written authorization from the account owner. This authorization note should name the auditor, the requested date range, the business reason for the request, and the person who approved access.

The approved delivery path is the secure customer portal. Support should use the portal link, expiry controls, and audit note required by the data-sharing workflow. Files that contain personal data must not be shared through public links or copied into ticket comments.


# 6. BM25: keyword search

Embeddings match **meaning**. BM25 matches **words**. A chunk's score is built from:

- **rare words count more**
- **repeating a word helps, but less each time**
- **long chunks are penalised**

`BM25Retriever` is used exactly like the vector retriever: question in, ranked chunks out.

In [18]:
import re
from langchain_community.retrievers import BM25Retriever


def tokenize(text):
    # lowercase words (the default just splits on spaces and keeps capitals and punctuation)
    return re.findall(r"\w+", text.lower())


toy_docs = [Document(page_content=t) for t in [
    "Support must verify written authorization before sharing the export file.",
    "Refund review is available within 14 days of renewal.",
    "Export files expire after 30 days.",
    "Never share the export file through public links.",
    "Password resets are handled by the identity provider administrator.",
]]

toy_retriever = BM25Retriever.from_documents(toy_docs, preprocess_func=tokenize, k=5)

for rank, doc in enumerate(toy_retriever.invoke("written authorization export"), 1):
    print(f"{rank}. {doc.page_content}")

1. Support must verify written authorization before sharing the export file.
2. Export files expire after 30 days.
3. Never share the export file through public links.
4. Password resets are handled by the identity provider administrator.
5. Refund review is available within 14 days of renewal.


/tmp/ipykernel_25310/325672880.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Now on the handbook chunks, with our question:

In [19]:
bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=CANDIDATE_K)

print("BM25 top 3")
show(bm25_retriever.invoke(FOCUS_QUESTION)[:3])
print("\nEmbedding top 3")
show(naive_context)

BM25 top 3
 1. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 2. chunk  1 | 'Support specialists should answer from the policy evidence that is actually retrieved. If '
 3. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'

Embedding top 3
 1. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 2. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'
 3. chunk  2 | '## Customer data exports\n\nSupport agents can request a customer data export from the works'


# 7. Hybrid search

Use both retrievers. `EnsembleRetriever` runs them and blends the results with a **weight** for each:

- `weights = [keyword weight, semantic weight]`
- our `alpha` is the **semantic** weight, so `weights = [1 − alpha, alpha]`
- `alpha = 1` is pure embeddings, `alpha = 0` is pure BM25

It blends the **rankings**, not the scores (reciprocal rank fusion): a chunk earns `weight / (rank + 60)` from each retriever, and the points are added up.

In [21]:
FOCUS_QUESTION

"A customer's outside reviewer is doing a quarterly controls check and asks the support agent for the workspace records package. The agent wants to paste a download link into the ticket thread so the reviewer can grab the file. What should support verify before anything is shared, and what is the approved delivery path?"

In [20]:
from langchain_classic.retrievers import EnsembleRetriever

for alpha in [0.0, 0.35, 0.65, 0.8, 1.0]:
    hybrid = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever], weights=[1 - alpha, alpha])
    print(f"alpha = {alpha}")
    show(hybrid.invoke(FOCUS_QUESTION)[:FINAL_CONTEXT_K])
    print()

alpha = 0.0
 1. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 2. chunk  1 | 'Support specialists should answer from the policy evidence that is actually retrieved. If '
 3. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'

alpha = 0.35
 1. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 2. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'
 3. chunk  1 | 'Support specialists should answer from the policy evidence that is actually retrieved. If '

alpha = 0.65
 1. chunk 14 | '## Data sharing and auditor requests\n\nExport links are for internal review unless the cust'
 2. chunk  3 | 'Before starting an export, support must confirm the workspace slug, requester email, and b'
 3. chunk 15 | 'Legal requests, subpoenas, and law-enforcement requests must move to Legal

# 8. Query rewriting (HyDE)

Sometimes the problem is the **wording**: the question does not sound like the handbook.

**HyDE**: ask the LLM to write a short *hypothetical* handbook passage that would answer the question, then search with that passage.

`HypotheticalDocumentEmbedder` wraps our embedding model: when it embeds a **question**, it first asks the LLM for the hypothetical passage and embeds that instead.

The passage is only used to **search**. The final answer is built from the **real chunks** it finds.

In [22]:
INCIDENT_QUESTION = (
    "An Enterprise customer says they may have lost production data and need a launch unblocked today. "
    "What priority and response promise should support apply, and what should they avoid promising?"
)

print("Normal search")
show(vector_retriever.invoke(INCIDENT_QUESTION)[:FINAL_CONTEXT_K])

Normal search
 1. chunk  8 | '## Enterprise response commitments\n\nEnterprise customers receive onboarding support, accou'
 2. chunk  9 | 'The response commitment measures first human response, not final resolution. Support shoul'
 3. chunk  7 | 'Service credits can be applied only by billing operations after a documented incident, dup'


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import HypotheticalDocumentEmbedder

HYDE_HINT = (
    "HelioDesk policy language may include Enterprise, severity-one, critical support tickets, "
    "first response, business hour, data loss, production outage, named escalation contact, "
    "incident commander, and resolution estimate."
)

hyde_prompt = PromptTemplate.from_template(
    "Write a short passage (2-3 sentences) in the style of a formal B2B support-policy handbook "
    "that would answer this question. Use the vocabulary a policy document would use. "
    "Do not address the customer or invent citations.\n"
    "Possible policy terms: {hint}\n\n"
    "Question: {question}\n\nPassage:"
).partial(hint=HYDE_HINT)

hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(llm, embeddings, custom_prompt=hyde_prompt)

# what the LLM writes for our question (search text only, never evidence)
print(hyde_embeddings.llm_chain.invoke({"question": INCIDENT_QUESTION}))

In the event of a reported Enterprise data loss incident impacting production, support shall classify the ticket as severity-one, triggering immediate engagement. A first response within one business hour is mandated, with an incident commander assigned to coordinate efforts and provide a preliminary resolution estimate. Support must avoid promising specific resolution times or guaranteeing data recovery, focusing instead on the mitigation and restoration process.


Same stored vectors, but questions are now embedded the HyDE way:

In [24]:
hyde_store = Chroma(client=chroma_client, collection_name="heliodesk", embedding_function=hyde_embeddings)
hyde_retriever = hyde_store.as_retriever(search_kwargs={"k": CANDIDATE_K})

print("Search with the hypothetical passage")
show(hyde_retriever.invoke(INCIDENT_QUESTION)[:FINAL_CONTEXT_K])

Search with the hypothetical passage
 1. chunk  8 | '## Enterprise response commitments\n\nEnterprise customers receive onboarding support, accou'
 2. chunk  9 | 'The response commitment measures first human response, not final resolution. Support shoul'
 3. chunk  7 | 'Service credits can be applied only by billing operations after a documented incident, dup'


HyDE together with hybrid search: BM25 uses the original question, the embedding side uses the passage.

In [25]:
hyde_hybrid = EnsembleRetriever(retrievers=[bm25_retriever, hyde_retriever], weights=[0.35, 0.65])
hyde_context = hyde_hybrid.invoke(INCIDENT_QUESTION)[:FINAL_CONTEXT_K]
show(hyde_context)

 1. chunk  9 | 'The response commitment measures first human response, not final resolution. Support shoul'
 2. chunk  8 | '## Enterprise response commitments\n\nEnterprise customers receive onboarding support, accou'
 3. chunk 17 | 'If a support specialist accidentally exposes customer data in a ticket, macro, screenshot,'


In [ ]:
print(answer(INCIDENT_QUESTION, hyde_context))

# Recap

| The problem | The lever |
|---|---|
| The right chunk is found but ranked too low | **Reranking** |
| The right chunk shares exact words with the question (or the reverse) | **Hybrid search** |
| The question is worded differently from the handbook | **Query rewriting (HyDE)** |

Each lever was only a few lines:

| Block | LangChain | Course helper core |
|---|---|---|
| Embeddings | `OpenAIEmbeddings` | `OpenRouterEmbedder` |
| Vector database | `Chroma` | `ChromaStore` |
| Keyword search | `BM25Retriever` | `BM25Retriever` |
| Hybrid search | `EnsembleRetriever` | `HybridRetriever` |
| Reranking | HTTP call to OpenRouter | `OpenRouterReranker` |
| Query rewriting | `HypotheticalDocumentEmbedder` | `HyDERewriter` |